In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
from scipy.stats import gaussian_kde
from statsmodels.stats.multitest import fdrcorrection
from pathlib import Path
from typing import Tuple, Optional

# ==============================================================================
# 1. Core Analytical Function: Group-Level Regression Coefficients
# ==============================================================================
def analyze_hddm_group_regression(
    nc_path: Path, 
    output_dir: Path, 
    parameter_prefix: str = 'v_'
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """
    Extracts group-level regression coefficients (excluding subject-specific random 
    effects) from HDDM posterior traces. Computes Bayesian inference statistics, 
    including High-Density Intervals (HDI), two-sided Bayesian p-values, and 
    False Discovery Rate (FDR) corrections.
    """
    print(f"[INFO] Loading posterior traces from: {nc_path}")
    idata = az.from_netcdf(nc_path)
    
    # Core filtering logic: Retain parameters matching the prefix and exclude subject-level effects
    group_vars = [
        v for v in idata.posterior.data_vars 
        if v.startswith(parameter_prefix) and 'subj' not in v
    ]
    
    if not group_vars:
        print(f"[WARNING] No group-level parameters found matching prefix: '{parameter_prefix}'")
        return None, None
    
    posterior_df = idata.posterior[group_vars].to_dataframe().reset_index()
    
    results = []
    # 1. 常规参数提取
    for var in group_vars:
        trace = posterior_df[var].values
        mean_val = np.mean(trace)
        hdi = az.hdi(trace, hdi_prob=0.95)
        
        # Calculate two-sided Bayesian P-value
        p_val = 2 * min(np.mean(trace > 0), np.mean(trace < 0))
        
        results.append({
            'Parameter': var,
            'Mean': mean_val,
            '95%_HDI_Lower': hdi[0],
            '95%_HDI_Upper': hdi[1],
            'P_value': p_val
        })
    
    # ==============================================================================
    # 2. 新增模块：计算 Within-Category 条件下的简单效应 (Continuous vs Categorical)
    # ==============================================================================
    # 根据你上一轮的 Patsy 编码，精准匹配这两个列名
    train_eff_col = "v_C(judgment_condition, Treatment('categorical'))[T.continuous]"
    inter_eff_col = "v_C(judgment_condition, Treatment('categorical'))[T.continuous]:C(pair_type, Treatment('between'))[T.within]"
    
    if train_eff_col in posterior_df.columns and inter_eff_col in posterior_df.columns:
        print("[INFO] Computing custom simple effect for Within-Category pairs...")
        
        # 将两条后验链相加 (主效应 + 交互作用)
        trace_within_diff = posterior_df[train_eff_col].values + posterior_df[inter_eff_col].values
        
        # 计算新链的统计量
        mean_val_custom = np.mean(trace_within_diff)
        hdi_custom = az.hdi(trace_within_diff, hdi_prob=0.95)
        p_val_custom = 2 * min(np.mean(trace_within_diff > 0), np.mean(trace_within_diff < 0))
        
        # 为这个简单效应命名
        custom_param_name = "v_Simple_Effect(Continuous_vs_Categorical_at_Within)"
        
        # 将新生成的链放回 dataframe，这样你的画图函数也能直接把它画出来
        posterior_df[custom_param_name] = trace_within_diff
        
        # 记录到结果列表中
        results.append({
            'Parameter': custom_param_name,
            'Mean': mean_val_custom,
            '95%_HDI_Lower': hdi_custom[0],
            '95%_HDI_Upper': hdi_custom[1],
            'P_value': p_val_custom
        })
    # ==============================================================================
    
    stats_df = pd.DataFrame(results)
    
    # Apply Benjamini-Hochberg FDR correction for multiple comparisons
    rejected, p_fdr = fdrcorrection(stats_df['P_value'])
    stats_df['P_fdr'] = p_fdr
    stats_df['Significant'] = rejected
    
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / f"Group_Testing_{parameter_prefix}.csv"
    stats_df.to_csv(save_path, index=False)
    
    print(f"[SUCCESS] Group regression statistics exported to: {save_path}")
    return stats_df, posterior_df
# ==============================================================================
# 2. Visualization Function: Group Posterior Distributions
# ==============================================================================
def plot_group_posteriors(
    posterior_df: pd.DataFrame, 
    stats_df: pd.DataFrame, 
    output_dir: Path, 
    parameter_prefix: str = 'v_'
) -> None:
    """
    Visualizes the posterior density distributions of group-level parameters.
    Applies Y-axis normalization (peak fixed at 1.0) and dynamically annotates 
    FDR-corrected significance.

    Args:
        posterior_df (pd.DataFrame): Raw posterior traces for the target parameters.
        stats_df (pd.DataFrame): Summary statistics containing FDR-corrected p-values.
        output_dir (Path): Destination directory for the exported plots.
        parameter_prefix (str): The prefix used to clean parameter names for titles.
    """
    vars_to_plot = stats_df['Parameter'].tolist()
    n_vars = len(vars_to_plot)
    
    # Determine dynamic grid layout
    cols = min(2, n_vars)
    rows = int(np.ceil(n_vars / cols))
    
    # Explicitly specify canvas dimensions to prevent rendering errors
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 6 * rows), squeeze=False)
    axes_flat = axes.flatten()
    
    sns.set_theme(style="white", font_scale=1.5)
    
    for i, var in enumerate(vars_to_plot):
        ax = axes_flat[i]
        trace = posterior_df[var].values
        row = stats_df[stats_df['Parameter'] == var].iloc[0]
        
        # Compute normalized Kernel Density Estimation (KDE)
        kde = gaussian_kde(trace)
        x = np.linspace(trace.min() - np.std(trace), trace.max() + np.std(trace), 1000)
        y = kde(x)
        y_norm = y / np.max(y)  # Normalize peak density to 1.0
        
        # Color coding: Purple for significant effects (FDR < 0.05), Grey for null effects
        color = "#6A51A3" if row['P_fdr'] < 0.05 else "#969696"
        
        ax.plot(x, y_norm, color=color, linewidth=4)
        ax.fill_between(x, 0, y_norm, color=color, alpha=0.3)
        
        # Add vertical reference line at zero (null hypothesis)
        ax.axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.6)
        
        # Standardize Y-axis limits and ticks
        ax.set_ylim(0, 1.1)
        ax.set_yticks([0, 0.5, 1.0])
        ax.set_yticklabels(['0', '0.5', '1.0'])
        
        # Format publication-ready titles (stripping Patsy notation)
        title = var.replace(parameter_prefix, "").replace("type[T.", " ").replace("]", "").replace(":", " x ")
        
        # LaTeX formatting applied for mathematical notation within Matplotlib titles
        ax.set_title(f"{title}\n$P_{{fdr}}$ = {row['P_fdr']:.4f}", fontweight='bold', fontsize=22)
        
        for spine in ['bottom', 'left']:
            ax.spines[spine].set_linewidth(3.0)
        sns.despine(ax=ax)

    # Dynamically remove surplus subplots in the grid
    for j in range(i + 1, len(axes_flat)):
        fig.delaxes(axes_flat[j])

    plt.tight_layout()
    
    # Robust export logic with fallback mechanism
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / f"Group_Posteriors_{parameter_prefix.strip('_')}.png"
    
    try:
        fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"[SUCCESS] Group-level posterior distribution plot exported to: {save_path}")
    except Exception as e:
        print(f"[ERROR] Plot export failed. Attempting fallback save method. Exception: {e}")
        plt.savefig(save_path)
    finally:
        plt.close(fig)


# ==============================================================================
# Execution Pipeline
# ==============================================================================
if __name__ == "__main__":
    # Define explicit analytical paths
    RESULT_DIR = Path("../../results/hddm/posterior/M4_v_judgment_x_pair")
    NC_PATH = Path("../../reference_outputs/hddm/formal_models/M4_v_judgment_x_pair/M4_v_judgment_x_pair_fitted.nc")

    print(f"\n{'='*70}")
    print("[INFO] Initiating Group-Level Regression Analysis")
    print(f"{'='*70}")

    # Extract statistics and traces for drift rate ('v_')
    v_stats, v_samples = analyze_hddm_group_regression(NC_PATH, RESULT_DIR, 'v_')

    # Execute plotting sequence if parameters were successfully resolved
    if v_stats is not None and v_samples is not None:
        print("[INFO] Initiating visualization rendering for drift rate ('v_') parameters...")
        plot_group_posteriors(v_samples, v_stats, RESULT_DIR, 'v_')
        
    print("[SUCCESS] Analytical sequence concluded.")

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
from pathlib import Path
from typing import Tuple, List, Optional

# ==============================================================================
# Posterior Reconstruction and Effect Derivation Module
# ==============================================================================

def reconstruct_condition_v(
    nc_path: Path, 
    output_dir: Path
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reconstructs the drift rate (v) posterior distributions for all 4 experimental 
    conditions from the fitted regression coefficients. Simultaneously derives 
    all simple effects, main effects, and interaction terms directly from the 
    joint posterior, calculating summary statistics (Mean, SD, 95% HDI, P(>0)).

    Args:
        nc_path (Path): Path to the serialized NetCDF model traces.
        output_dir (Path): Destination directory for the exported CSV tables.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: A tuple containing the summary statistics 
        dataframe and the full posterior samples dataframe for all conditions/effects.
    """
    print("[INFO] Reconstructing condition means and derived effect statistics from regression coefficients...")
    
    idata = az.from_netcdf(nc_path)
    post = idata.posterior
    all_vars = list(post.data_vars.keys())
    
    def find_var(keywords: List[str], exclude: Optional[List[str]] = None) -> Optional[str]:
        """Helper function to dynamically locate parameter names in the trace."""
        for v in all_vars:
            if all(k in v for k in keywords):
                if exclude and any(e in v for e in exclude):
                    continue
                return v
        return None

    # 1. Automatically map regression coefficients based on Patsy naming conventions
    name_b0 = find_var(['v_', 'Intercept'])
    name_cont = find_var(['v_', 'judgment', 'continuous'], exclude=[':'])
    name_wit = find_var(['v_', 'pair', 'within'], exclude=[':'])
    name_int = find_var(['v_', ':', 'continuous', 'within'])

    if not all([name_b0, name_cont, name_wit, name_int]):
        raise KeyError(
            "[ERROR] Failed to match all required regression coefficients. "
            "Please verify the model formula and trace variables."
        )

    b0 = post[name_b0].values.flatten()
    b_cont = post[name_cont].values.flatten()
    b_wit = post[name_wit].values.flatten()
    b_int = post[name_int].values.flatten()
    
    # 2. Reconstruct the four base experimental conditions
    v_cat_bet = b0
    v_cat_wit = b0 + b_wit
    v_cont_bet = b0 + b_cont
    v_cont_wit = b0 + b_cont + b_wit + b_int
    
    reconstructed_df = pd.DataFrame({
        'categorical_between': v_cat_bet,
        'categorical_within': v_cat_wit,
        'continuous_between': v_cont_bet,
        'continuous_within': v_cont_wit
    })
    
    # 3. Compute derived effects directly within the joint posterior distribution
    
    # -- Simple Effects --
    reconstructed_df['simple_judgment_between'] = v_cat_bet - v_cont_bet
    reconstructed_df['simple_judgment_within'] = v_cat_wit - v_cont_wit
    reconstructed_df['simple_pair_categorical'] = v_cat_bet - v_cat_wit
    reconstructed_df['simple_pair_continuous'] = v_cont_bet - v_cont_wit
    
    # -- Main Effects --
    reconstructed_df['main_judgment'] = ((v_cat_bet + v_cat_wit) / 2) - ((v_cont_bet + v_cont_wit) / 2)
    reconstructed_df['main_pair'] = ((v_cat_bet + v_cont_bet) / 2) - ((v_cat_wit + v_cont_wit) / 2)
    
    # -- Interaction Effect --
    reconstructed_df['interaction'] = (v_cat_bet - v_cat_wit) - (v_cont_bet - v_cont_wit)
    
    # 4. Generate Comprehensive Summary Table
    # Calculate Mean and Standard Deviation
    summary = reconstructed_df.agg(['mean', 'std']).T
    
    # Calculate 95% Highest Density Interval (HDI) and Directional Probability P(>0)
    summary['hdi_2.5%'] = reconstructed_df.apply(lambda x: az.hdi(x.values, hdi_prob=0.95)[0])
    summary['hdi_97.5%'] = reconstructed_df.apply(lambda x: az.hdi(x.values, hdi_prob=0.95)[1])
    summary['P(>0)'] = (reconstructed_df > 0).mean()
    
    # Format to 4 decimal places for publication-ready manuscript tables
    summary = summary.round(4)
    
    # 5. Export Results
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    samples_save_path = output_dir / "v_conditions_and_effects_samples.csv"
    summary_save_path = output_dir / "v_conditions_summary_with_effects.csv"
    
    reconstructed_df.to_csv(samples_save_path, index=False)
    summary.to_csv(summary_save_path)
    
    print(f"[SUCCESS] Reconstruction complete. Comprehensive summary table saved to: {summary_save_path}")
    
    return summary, reconstructed_df


# ==============================================================================
# Execution Pipeline
# ==============================================================================
if __name__ == "__main__":
    # Example execution paths (Modify as needed for your environment)
    # RESULT_DIR = Path("../../results/hddm/posterior/M4_v_judgment_x_pair")
    # NC_PATH = Path("../../reference_outputs/hddm/formal_models/M4_v_judgment_x_pair/M4_v_judgment_x_pair_fitted.nc")
    
    # v_summary, v_samples = reconstruct_condition_v(NC_PATH, RESULT_DIR)

    # print("\n[INFO] Comprehensive Statistics Table (Means and Derived Effects):")
    # print(v_summary)
    pass

In [ ]:
from pathlib import Path

# ==============================================================================
# Execution Block: Drift Rate (v) Posterior Reconstruction
# ==============================================================================

if __name__ == "__main__":
    # Define explicit I/O paths using pathlib for cross-platform robustness
    NC_PATH = Path("../../reference_outputs/hddm/formal_models/M4_v_judgment_x_pair/M4_v_judgment_x_pair_fitted.nc")
    RESULT_DIR = Path("../../results/hddm/posterior/M4_v_judgment_x_pair")

    print(f"\n{'='*70}")
    print("[INFO] Initiating Drift Rate (v) Reconstruction Pipeline")
    print(f"{'='*70}")

    # Execute the reconstruction and derivation function
    v_summary, v_samples = reconstruct_condition_v(
        nc_path=NC_PATH, 
        output_dir=RESULT_DIR
    )

    # Display the comprehensive statistical summary in the console
    print("\n[INFO] Comprehensive Statistical Summary (Condition Means and Derived Effects):")
    print(v_summary)
    
    print("\n[SUCCESS] Reconstruction pipeline executed successfully.")